In [ ]:
from pathlib import Path
import os
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root))
from experiments.paths import GENERATED, PROJECT_ROOT

run_directory = GENERATED / 'soay/age_size/validation/gp_accuracy'
run_directory.mkdir(parents=True, exist_ok=True)
os.chdir(run_directory)


In [ ]:
from pathlib import Path
import sys
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = PROJECT_ROOT

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from experiments.soay.age_size.abc.deeprv.deeprv_decoder_agesize import get_runtime

checkpoint_folder = GENERATED / 'soay/age_size/training/transformer'
checkpoint_path = checkpoint_folder / "model.ckpt"
data_path = GENERATED / 'soay/age_size/data/gp_ibm_dataset_agesize.csv'
samples_path = GENERATED / 'soay/age_size/inference/gpdf_agesize_gp_samples.npz'
output_directory = checkpoint_folder / "evaluation_outputs"

output_directory.mkdir(exist_ok=True)

decoder = get_runtime(checkpoint_path)
gp_data = pd.read_csv(data_path)

with np.load(samples_path) as saved:
    posterior_lengthscales = {
        "Survival": saved["surv_ls"].copy(),
        "Reproduction": saved["repr_ls"].copy(),
        "Growth": saved["growth_ls"].copy(),
    }

print(f"Checkpoint: {checkpoint_path}")
print(f"Maximum Locations: {decoder.max_locations}")
print(f"JAX Devices: {jax.devices()}")

In [ ]:
vital_rate_order = ["Survival", "Reproduction", "Growth"]
years = np.sort(gp_data["yr"].dropna().unique())

vital_rate_masks = {
    "Survival": gp_data[["z", "age", "Surv"]].notna().all(axis=1),
    "Reproduction": gp_data[["z", "age", "Repr"]].notna().all(axis=1) & gp_data["age"].ge(1),
    "Growth": gp_data[["z", "age", "z1"]].notna().all(axis=1),
}

yearly_locations = {}

for vital_rate in vital_rate_order:
    yearly_locations[vital_rate] = {}

    for year in years:
        mask = vital_rate_masks[vital_rate] & gp_data["yr"].eq(year)
        locations = gp_data.loc[mask, ["z", "age"]].to_numpy(dtype=np.float32)

        if len(locations) > 0:
            if len(locations) > decoder.max_locations:
                raise ValueError(f"{vital_rate}, Year {year}: {len(locations)} locations exceeds {decoder.max_locations}")

            yearly_locations[vital_rate][year] = locations

    counts = {int(year): len(locations) for year, locations in yearly_locations[vital_rate].items()}
    print(f"{vital_rate}: {counts}")

In [ ]:
def ard_rbf_covariance(locations, lengthscales):
    locations = np.asarray(locations, dtype=np.float32)
    lengthscales = np.asarray(lengthscales, dtype=np.float32)
    differences = (locations[:, None, :] - locations[None, :, :]) / lengthscales
    return np.exp(-0.5 * np.sum(differences**2, axis=-1)).astype(np.float32)


def evaluate_age_size_draws(locations, lengthscales, latent_z):
    locations = np.asarray(locations, dtype=np.float32)
    lengthscales = np.asarray(lengthscales, dtype=np.float32).reshape(2)
    latent_z = np.asarray(latent_z, dtype=np.float32)

    number_draws, number_locations = latent_z.shape

    if locations.shape != (number_locations, 2):
        raise ValueError("Locations must have shape (number_locations, 2)")

    padded_locations = np.zeros((decoder.max_locations, 2), dtype=np.float32)
    padded_locations[:number_locations] = locations

    padded_z = np.zeros((number_draws, decoder.max_locations), dtype=np.float32)
    padded_z[:, :number_locations] = latent_z

    mask = np.arange(decoder.max_locations) < number_locations
    padded_covariance = ard_rbf_covariance(padded_locations, lengthscales)
    padded_covariance += decoder.jitter * np.eye(decoder.max_locations, dtype=np.float32)

    active_covariance = ard_rbf_covariance(locations, lengthscales)
    active_covariance += decoder.jitter * np.eye(number_locations, dtype=np.float32)

    exact = np.einsum("ij,bj->bi", np.linalg.cholesky(active_covariance), latent_z)

    predicted = decoder.decode(
        jnp.asarray(padded_z),
        jnp.asarray(lengthscales),
        jnp.asarray(padded_locations),
        jnp.asarray(padded_covariance),
        jnp.repeat(jnp.asarray(mask)[None], number_draws, axis=0),
    )[:, :number_locations, 0]

    predicted.block_until_ready()
    return exact, np.asarray(jax.device_get(predicted))

In [ ]:
number_lengthscale_samples = 5000
draws_per_lengthscale = 10
evaluation_seed = 20260825

evaluation_rng = np.random.default_rng(evaluation_seed)
result_rows = []

for vital_rate in vital_rate_order:
    print(f"Evaluating {vital_rate}")

    all_lengthscales = np.asarray(posterior_lengthscales[vital_rate], dtype=np.float32)
    valid = np.isfinite(all_lengthscales).all(axis=1) & (all_lengthscales > 0).all(axis=1)
    valid_indices = np.flatnonzero(valid)

    if number_lengthscale_samples is None or number_lengthscale_samples >= len(valid_indices):
        selected_indices = valid_indices
    else:
        selected_indices = evaluation_rng.choice(valid_indices, size=number_lengthscale_samples, replace=False)

    for position, sample_index in enumerate(selected_indices):
        lengthscales = all_lengthscales[sample_index]
        yearly_mse = []

        for year, locations in yearly_locations[vital_rate].items():
            latent_z = evaluation_rng.standard_normal((draws_per_lengthscale, len(locations))).astype(np.float32)
            exact, predicted = evaluate_age_size_draws(locations, lengthscales, latent_z)
            yearly_mse.append(np.mean((exact - predicted)**2))

        mean_mse = float(np.mean(yearly_mse))

        result_rows.append({
            "Vital Rate": vital_rate,
            "MCMC Sample": int(sample_index),
            "Size Lengthscale": float(lengthscales[0]),
            "Age Lengthscale": float(lengthscales[1]),
            "Mean MSE": mean_mse,
            "RMSE": np.sqrt(mean_mse),
            "Draws Per Year": draws_per_lengthscale,
            "Number Of Years": len(yearly_mse),
        })

        if (position + 1) % 250 == 0 or position + 1 == len(selected_indices):
            print(f"  Completed {position + 1:,}/{len(selected_indices):,}")

age_size_results = pd.DataFrame(result_rows)

print()
print(f"Posterior Samples Per Vital Rate: {age_size_results.groupby('Vital Rate').size().to_dict()}")
print(f"GP–DeepRV Draws Per Year: {draws_per_lengthscale}")
display(age_size_results.head())

In [ ]:
plt.style.use("seaborn-v0_8-white")

figure, axes = plt.subplots(
    3,
    2,
    figsize=(5.906, 6.4),
    sharey=True,
    constrained_layout=True,
)

for row, vital_rate in enumerate(vital_rate_order):
    current = age_size_results[age_size_results["Vital Rate"] == vital_rate]

    axes[row, 0].scatter(
        current["Size Lengthscale"],
        current["RMSE"],
        color="#0072B2",
        s=6,
        alpha=0.25,
        rasterized=True,
    )

    axes[row, 1].scatter(
        current["Age Lengthscale"],
        current["RMSE"],
        color="#D55E00",
        s=6,
        alpha=0.25,
        rasterized=True,
    )

    for axis in axes[row]:
        axis.set_xscale
        axis.set_ylim(bottom=0)
        axis.grid(False)
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)
        axis.tick_params(labelsize=9, direction="out")

    axes[row, 0].set_ylabel(f"{vital_rate}\nRMSE", fontsize=10)

axes[0, 0].set_title("Size Lengthscale", fontsize=11)
axes[0, 1].set_title("Age Lengthscale", fontsize=11)
axes[-1, 0].set_xlabel("Posterior Size Lengthscale", fontsize=10)
axes[-1, 1].set_xlabel("Posterior Age Lengthscale", fontsize=10)

age_size_results.to_csv(output_directory / "age_size_rmse_across_posterior_lengthscales.csv", index=False)
figure.savefig(output_directory / "age_size_rmse_across_posterior_lengthscales.pdf", bbox_inches="tight")
figure.savefig(output_directory / "age_size_rmse_across_posterior_lengthscales.png", dpi=300, bbox_inches="tight")

plt.show()

print(f"Saved Outputs: {output_directory}")

In [ ]:
from matplotlib.colors import Normalize
from matplotlib.colors import LinearSegmentedColormap

plt.style.use("seaborn-v0_8-white")

blue_cmap = LinearSegmentedColormap.from_list(
    "Dissertation Blue",
    ["#DCEAF4", "#56B4E9", "#0072B2", "#003B5C"],
)
colour_scale = Normalize(vmin=0, vmax=age_size_results["RMSE"].max())

figure, axes = plt.subplots(
    1,
    3,
    figsize=(5.906, 2.7),
    sharex=False,
    sharey=False,
    constrained_layout=True,
)

for axis, vital_rate in zip(axes, vital_rate_order):
    current = age_size_results[age_size_results["Vital Rate"] == vital_rate]

    points = axis.scatter(
        current["Size Lengthscale"],
        current["Age Lengthscale"],
        c=current["RMSE"],
        cmap="viridis",        
        norm=colour_scale,
        s=12,
        alpha=0.75,
        edgecolors="none",
        rasterized=True,
    )

    axis.set_xlabel("Size Lengthscale", fontsize=10)
    axis.set_ylabel("Age Lengthscale", fontsize=10)
    axis.set_title(vital_rate, fontsize=11)
    axis.grid(False)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.tick_params(labelsize=9, direction="out")

colourbar = figure.colorbar(points, ax=axes, pad=0.02)
colourbar.set_label("RMSE", fontsize=10)
colourbar.ax.tick_params(labelsize=9)

figure.savefig(output_directory / "age_size_joint_lengthscales_rmse.pdf", bbox_inches="tight")
figure.savefig(output_directory / "age_size_joint_lengthscales_rmse.png", dpi=300, bbox_inches="tight")

plt.show()